# 2. split

## 1) 경로/설정 - 64/16 split

In [1]:
import random, shutil
from pathlib import Path

# ====== 너 폴더에 맞게 수정 ======
SRC_DIR   = Path(r"C:\gachikium\dataset\train_copy")  #  원본 보존
TRAIN_DIR = Path(r"C:\gachikium\dataset\train")       
TEST_DIR  = Path(r"C:\gachikium\dataset\test")        

TRAIN_RATIO = 0.8
SEED = 42
CLEAR_OUTPUT = True  # ✅ train/test 폴더를 매번 깨끗하게 다시 생성(원본은 건드리지 않음)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".jfif"}
random.seed(SEED)

## 2) split 실행

In [2]:
def clear_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def list_images(folder: Path):
    return [p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in IMG_EXTS]

def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    # 혹시 같은 파일명이 들어오면 덮어쓰지 않게 _1, _2 붙임
    if dst.exists():
        stem, suf = dst.stem, dst.suffix
        i = 1
        while (dst.parent / f"{stem}_{i}{suf}").exists():
            i += 1
        dst = dst.parent / f"{stem}_{i}{suf}"
    shutil.copy2(src, dst)

def split_train_test(paths, train_ratio=0.8):
    paths = list(paths)
    random.shuffle(paths)
    n = len(paths)
    n_train = int(n * train_ratio)  # 80이면 64
    if n >= 2:
        n_train = min(max(n_train, 1), n - 1)
    return paths[:n_train], paths[n_train:]

# ---- 실행 ----
if not SRC_DIR.exists():
    raise FileNotFoundError(f"SRC_DIR not found: {SRC_DIR}")

class_dirs = [d for d in SRC_DIR.iterdir() if d.is_dir()]
if not class_dirs:
    raise RuntimeError(f"No class folders found in: {SRC_DIR}")

if CLEAR_OUTPUT:
    clear_dir(TRAIN_DIR)
    clear_dir(TEST_DIR)

print("=== SPLIT START ===")
print("SRC  :", SRC_DIR)
print("TRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)
print(f"ratio={TRAIN_RATIO}, seed={SEED}\n")

for cls_dir in sorted(class_dirs, key=lambda x: x.name):
    imgs = list_images(cls_dir)
    if not imgs:
        print(f"[WARN] {cls_dir.name}: no images")
        continue

    # ✅ 안전장치: 80장이 아닐 때 바로 알려줌(그대로 split은 하되 경고)
    if len(imgs) != 80:
        print(f"[WARN] {cls_dir.name}: expected 80 but got {len(imgs)}")

    train_list, test_list = split_train_test(imgs, TRAIN_RATIO)

    for src in train_list:
        safe_copy(src, TRAIN_DIR / cls_dir.name / src.name)

    for src in test_list:
        safe_copy(src, TEST_DIR / cls_dir.name / src.name)

    print(f"{cls_dir.name}: total={len(imgs)}, train={len(train_list)}, test={len(test_list)}")

print("\nDONE.")
print("Train:", TRAIN_DIR.resolve())
print("Test :", TEST_DIR.resolve())

=== SPLIT START ===
SRC  : C:\gachikium\dataset\train_copy
TRAIN: C:\gachikium\dataset\train
TEST : C:\gachikium\dataset\test
ratio=0.8, seed=42

강아지상: total=80, train=64, test=16
[WARN] 고양이상: expected 80 but got 68
고양이상: total=68, train=54, test=14
[WARN] 곰상: expected 80 but got 73
곰상: total=73, train=58, test=15
[WARN] 꼬부기상: expected 80 but got 70
꼬부기상: total=70, train=56, test=14
[WARN] 도롱뇽상: expected 80 but got 76
도롱뇽상: total=76, train=60, test=16
[WARN] 사슴상: expected 80 but got 63
사슴상: total=63, train=50, test=13
여우상: total=80, train=64, test=16
쥐상: total=80, train=64, test=16
[WARN] 쿼카상: expected 80 but got 79
쿼카상: total=79, train=63, test=16
[WARN] 토끼상: expected 80 but got 79
토끼상: total=79, train=63, test=16

DONE.
Train: C:\gachikium\dataset\train
Test : C:\gachikium\dataset\test


In [10]:
from pathlib import Path


folder = Path(r"C:\gachikium\dataset\train_copy\토끼상")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".jfif"}

images = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
images = sorted(images, key=lambda x: x.name)

# 1단계: 임시 이름으로 변경
temp_paths = []
for i, img in enumerate(images, start=1):
    temp_name = folder / f"__temp__{i:03d}{img.suffix.lower()}"
    img.rename(temp_name)
    temp_paths.append(temp_name)

# 2단계: 최종 이름으로 변경
for i, img in enumerate(sorted(temp_paths), start=1):
    new_name = folder / f"Image_{i:02d}{img.suffix.lower()}"
    img.rename(new_name)

print("이름 변경 완료")
print(f"총 파일 수: {len(images)}")

이름 변경 완료
총 파일 수: 80


In [11]:
import random
import shutil
from pathlib import Path

# 경로 설정
SRC_DIR   = Path(r"C:\gachikium\dataset\train_copy")
TRAIN_DIR = Path(r"C:\gachikium\dataset\train")
TEST_DIR  = Path(r"C:\gachikium\dataset\test")

TRAIN_RATIO = 0.8
SEED = 42

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

random.seed(SEED)

print("=== SPLIT START ===")
print("SRC  :", SRC_DIR)
print("TRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)

# train / test 폴더 초기화
if TRAIN_DIR.exists():
    shutil.rmtree(TRAIN_DIR)

if TEST_DIR.exists():
    shutil.rmtree(TEST_DIR)

TRAIN_DIR.mkdir(parents=True)
TEST_DIR.mkdir(parents=True)

# 클래스 폴더 반복
for cls_dir in sorted([d for d in SRC_DIR.iterdir() if d.is_dir()]):

    imgs = [p for p in cls_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    imgs = sorted(imgs)

    total = len(imgs)

    random.shuffle(imgs)

    n_train = int(total * TRAIN_RATIO)

    train_imgs = imgs[:n_train]
    test_imgs = imgs[n_train:]

    # train 복사
    for src in train_imgs:
        dst = TRAIN_DIR / cls_dir.name / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    # test 복사
    for src in test_imgs:
        dst = TEST_DIR / cls_dir.name / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    print(f"{cls_dir.name}: total={total}, train={len(train_imgs)}, test={len(test_imgs)}")

print("\nDONE.")
print("Train:", TRAIN_DIR)
print("Test :", TEST_DIR)

=== SPLIT START ===
SRC  : C:\gachikium\dataset\train_copy
TRAIN: C:\gachikium\dataset\train
TEST : C:\gachikium\dataset\test
강아지상: total=80, train=64, test=16
고양이상: total=80, train=64, test=16
곰상: total=80, train=64, test=16
꼬부기상: total=80, train=64, test=16
도롱뇽상: total=80, train=64, test=16
사슴상: total=80, train=64, test=16
여우상: total=80, train=64, test=16
쥐상: total=80, train=64, test=16
쿼카상: total=80, train=64, test=16
토끼상: total=80, train=64, test=16

DONE.
Train: C:\gachikium\dataset\train
Test : C:\gachikium\dataset\test


In [12]:
from pathlib import Path

TRAIN_DIR = Path(r"C:\gachikium\dataset\train")
TEST_DIR  = Path(r"C:\gachikium\dataset\test")

train_names = set([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
test_names  = set([d.name for d in TEST_DIR.iterdir() if d.is_dir()])

print("train only:", train_names - test_names)
print("test only :", test_names - train_names)
print("same      :", train_names == test_names)

train only: set()
test only : set()
same      : True


In [14]:
from pathlib import Path

TRAIN_DIR = Path(r"C:\gachikium\dataset\train")
TEST_DIR  = Path(r"C:\gachikium\dataset\test")

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".jfif"}

def count_images(folder):
    return len([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS])

train_total = 0
test_total = 0

print("=== TRAIN ===")
for cls in TRAIN_DIR.iterdir():
    if cls.is_dir():
        n = count_images(cls)
        train_total += n
        print(cls.name, n)

print("train total:", train_total)

print("\n=== TEST ===")
for cls in TEST_DIR.iterdir():
    if cls.is_dir():
        n = count_images(cls)
        test_total += n
        print(cls.name, n)

print("test total:", test_total)
print("all total:", train_total + test_total)

=== TRAIN ===
강아지상 64
고양이상 64
곰상 64
꼬부기상 64
도롱뇽상 64
사슴상 64
여우상 64
쥐상 64
쿼카상 64
토끼상 64
train total: 640

=== TEST ===
강아지상 16
고양이상 16
곰상 16
꼬부기상 16
도롱뇽상 16
사슴상 16
여우상 16
쥐상 16
쿼카상 16
토끼상 16
test total: 160
all total: 800


In [15]:
from torchvision import transforms

transform_train = transforms.Compose([
    transforms.Resize(256),            # 짧은 변 256
    transforms.CenterCrop(224),        # 가운데 224x224
    transforms.RandomHorizontalFlip(), # 좌우 뒤집기 (데이터 증강)
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [17]:
from pathlib import Path

TRAIN_DIR = Path(r"C:\gachikium\dataset\train")
TEST_DIR  = Path(r"C:\gachikium\dataset\test")

print("TRAIN exists:", TRAIN_DIR.exists())
print("TEST exists :", TEST_DIR.exists())

if TRAIN_DIR.exists():
    print("train classes:", [d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])

if TEST_DIR.exists():
    print("test classes:", [d.name for d in TEST_DIR.iterdir() if d.is_dir()])

TRAIN exists: True
TEST exists : True
train classes: ['강아지상', '고양이상', '곰상', '꼬부기상', '도롱뇽상', '사슴상', '여우상', '쥐상', '쿼카상', '토끼상']
test classes: ['강아지상', '고양이상', '곰상', '꼬부기상', '도롱뇽상', '사슴상', '여우상', '쥐상', '쿼카상', '토끼상']


In [18]:
import cv2, numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".jfif"}

TRAIN_DIR = Path(r"C:\gachikium\dataset\train")
TEST_DIR  = Path(r"C:\gachikium\dataset\test")

# 1) buffalo_l 로드
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))   # GPU면 0, CPU면 -1

# 2) ArcFace recognition 모델 꺼내기
rec = app.models.get("recognition")
if rec is None:
    raise RuntimeError(f"recognition model not found. keys={list(app.models.keys())}")

# 유니코드 경로 읽기
def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

# L2 정규화
def l2_normalize(x):
    return x / (np.linalg.norm(x) + 1e-12)

# 112x112 BGR 얼굴 이미지 -> 512차원 임베딩
def get_embedding_112_bgr(img_bgr_112):
    if hasattr(rec, "get_feat"):
        emb = rec.get_feat(img_bgr_112)
    else:
        raise RuntimeError(f"rec has no get_feat(). available: {dir(rec)}")

    emb = np.asarray(emb).reshape(-1)
    return l2_normalize(emb).astype(np.float32)

# train/test 폴더를 읽어서 임베딩 X, 라벨 y 생성
def load_split_embeddings(root_dir: Path):
    if not root_dir.exists():
        raise FileNotFoundError(f"Path not found: {root_dir}")

    X, y = [], []
    classes = sorted([d.name for d in root_dir.iterdir() if d.is_dir()])
    class_to_idx = {c: i for i, c in enumerate(classes)}

    for cls in classes:
        cls_dir = root_dir / cls
        files = sorted(
            [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS],
            key=lambda p: p.name
        )

        for p in files:
            img = imread_unicode(p)
            if img is None:
                print(f"[WARN] unreadable image: {p}")
                continue

            # ArcFace recognition은 112x112 입력 기준
            if img.shape[0] != 112 or img.shape[1] != 112:
                img = cv2.resize(img, (112, 112), interpolation=cv2.INTER_AREA)

            try:
                emb = get_embedding_112_bgr(img)
                X.append(emb)
                y.append(class_to_idx[cls])
            except Exception as e:
                print(f"[WARN] embedding failed: {p.name} / {e}")
                continue

    if len(X) == 0:
        raise ValueError(f"No valid embeddings found in {root_dir}")

    return np.vstack(X), np.array(y), classes

# 평가 함수
def evaluate(clf, X_test, y_test, classes):
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("test acc:", acc)

    print("\nclassification_report:")
    print(classification_report(y_test, pred, target_names=classes))

    print("\nconfusion_matrix:")
    print(confusion_matrix(y_test, pred))

# 데이터 준비
X_train, y_train, classes = load_split_embeddings(TRAIN_DIR)
X_test,  y_test,  _       = load_split_embeddings(TEST_DIR)

print("X_train:", X_train.shape, "X_test:", X_test.shape, "num_classes:", len(classes))

# 방법 1) Logistic Regression
def train_logistic(X_train, y_train):
    clf = LogisticRegression(max_iter=5000, n_jobs=-1)
    clf.fit(X_train, y_train)
    return clf

# 방법 2) Linear SVM
def train_linear_svm(X_train, y_train):
    clf = LinearSVC()
    clf.fit(X_train, y_train)
    return clf

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\user/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

Logistic Regression

In [19]:
clf_log = train_logistic(X_train, y_train)
evaluate(clf_log, X_test, y_test, classes)

test acc: 0.78125

classification_report:
              precision    recall  f1-score   support

        강아지상       0.56      0.62      0.59        16
        고양이상       0.91      0.62      0.74        16
          곰상       0.89      1.00      0.94        16
        꼬부기상       0.77      0.62      0.69        16
        도롱뇽상       0.68      0.94      0.79        16
         사슴상       0.83      0.62      0.71        16
         여우상       0.79      0.69      0.73        16
          쥐상       0.82      0.88      0.85        16
         쿼카상       0.94      0.94      0.94        16
         토끼상       0.74      0.88      0.80        16

    accuracy                           0.78       160
   macro avg       0.79      0.78      0.78       160
weighted avg       0.79      0.78      0.78       160


confusion_matrix:
[[10  0  0  0  2  0  2  1  0  1]
 [ 1 10  2  0  1  0  0  2  0  0]
 [ 0  0 16  0  0  0  0  0  0  0]
 [ 4  0  0 10  0  0  1  0  1  0]
 [ 1  0  0  0 15  0  0  0  0  0]
 [ 1  0  0  0  

c:\gachikium\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Linear SVM

In [20]:
clf_svm = train_linear_svm(X_train, y_train)
evaluate(clf_svm, X_test, y_test, classes)

test acc: 0.85625

classification_report:
              precision    recall  f1-score   support

        강아지상       0.79      0.69      0.73        16
        고양이상       0.82      0.88      0.85        16
          곰상       0.94      1.00      0.97        16
        꼬부기상       0.86      0.75      0.80        16
        도롱뇽상       0.75      0.94      0.83        16
         사슴상       1.00      0.88      0.93        16
         여우상       0.80      0.75      0.77        16
          쥐상       0.88      0.88      0.88        16
         쿼카상       0.88      0.94      0.91        16
         토끼상       0.88      0.88      0.88        16

    accuracy                           0.86       160
   macro avg       0.86      0.86      0.86       160
weighted avg       0.86      0.86      0.86       160


confusion_matrix:
[[11  0  0  0  2  0  1  0  0  2]
 [ 0 14  1  0  0  0  0  1  0  0]
 [ 0  0 16  0  0  0  0  0  0  0]
 [ 1  0  0 12  1  0  1  0  1  0]
 [ 1  0  0  0 15  0  0  0  0  0]
 [ 0  0  0  0  